# I

## Intro to Data Cleaning with Apache Spark


- Course focus: **data cleaning** with Apache Spark and Python
- Covers: what data cleaning is, why it matters, how to do it at scale with Spark

### What is data cleaning?

**Data cleaning** = preparing raw data for use in **processing pipelines**. (Pipelines will be covered later — for now, think "the series of steps that take raw data to finished analytics".)

Why it matters:

- If data isn't clean, it isn't **trustworthy**
- Dirty data breaks downstream pipelines in subtle, expensive-to-debug ways
- A necessary part of any **production** data system — not an optional polish step

**Typical data-cleaning tasks:**

- **Reformatting or replacing text** (standardising dates, fixing casing, stripping whitespace)
- **Performing calculations** on the data (deriving columns, unit conversions)
- **Removing garbage or incomplete data** (bad rows, nulls, out-of-range values)

### Why perform data cleaning with Spark?

Two recurring problems in any data-cleaning system:

1. **Performance** — handling large volumes
2. **Flow organisation** — co-ordinating multi-step pipelines

**On performance:**

Traditional languages (Perl, C++, standard SQL on a single box) are fine for small datasets. They fall over once the data stretches into **millions or billions of rows** — single-machine throughput just isn't enough.

Spark's edge: processing **scales out** across a cluster as the data grows. Capacity expands with the workload rather than being capped by one machine.

**On flow:**

Large-scale cleaning naturally splits into a **pipeline** of stages — read, validate, transform, deduplicate, enrich, write out. Spark handles all of that within a **single framework** — no need to glue together half a dozen separate tools.

### Data cleaning — example

A small motivating example. Given raw data:

| Name | Age (years) | City |
|---|---|---|
| … | … | … |

The requirement for the cleaned output:

- **First name and last name in separate columns** (split a single "Name" field)
- **Age in months** instead of years (unit conversion)
- A new **state** column derived from the city
- **Out-of-the-ordinary rows removed** (bad or garbage entries dropped)

All of that is achievable with Spark transformations in a single pipeline — the cleaned DataFrame then flows into whatever processing comes next.

### Spark schemas

A primary function of data cleaning is **verifying data is in the expected format**. Spark provides built-in support for that via **schemas**.

Important to lock in: a **schema** defines and validates the **number and types of columns** in a DataFrame — analogous to a schema in a database or XML.

**Supported field types include:**

- Integers, floats
- Strings
- Dates
- Arrays
- Map structures

**Two big benefits of defining a schema:**

1. **Correctness** — Spark can filter out non-conforming data **during read**, catching problems as early as possible
2. **Performance** — without a schema, Spark **infers** one by reading the data **twice** (once to figure out types, once to actually load). An explicit schema skips the inference pass → **one read** instead of two

### Example Spark schema

Defining a schema uses the `pyspark.sql.types` module.

```python
import pyspark.sql.types

# Build the schema as a StructType of StructFields
peopleSchema = pyspark.sql.types.StructType([
    # name, dataType, nullable
    pyspark.sql.types.StructField('name',    pyspark.sql.types.StringType(),  True),
    pyspark.sql.types.StructField('age',     pyspark.sql.types.IntegerType(), True),
    pyspark.sql.types.StructField('city',    pyspark.sql.types.StringType(),  True),
])
```

With shorter imports — the common idiom:

```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

peopleSchema = StructType([
    StructField('name', StringType(), True),
    StructField('age',  IntegerType(), True),
    StructField('city', StringType(), True),
])
```

Apply it to a read:

```python
people_df = spark.read.format('csv') \
    .load('people.csv', schema=peopleSchema)
```

Key points on `StructField`:

- **Field name** — the column name
- **Data type** — `StringType()`, `IntegerType()`, `DateType()`, `ArrayType(...)`, etc.
- **Nullable flag** — `True` if the column can contain nulls

Important to remember: `.load()` takes both the **filename** and the **schema** — that's where the schema actually gets applied against the data.

### Key takeaways — cheat sheet

**The data-cleaning pitch for Spark in one table:**

| Problem | Why Spark helps |
|---|---|
| Huge datasets exceed single-machine limits | Scales out across a cluster |
| Pipelines sprawl across many tools | Single framework for read, transform, write |
| Dirty data slips through on read | Explicit schema filters bad rows at load time |
| Schema inference is slow | Explicit schema → **one** read instead of two |

**Defining a schema — minimal template:**

```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField('col1', StringType(),  True),
    StructField('col2', IntegerType(), True),
    # ...
])

df = spark.read.format('csv').load('file.csv', schema=schema)
```

**Things to lock in:**

- Data cleaning = **preparing raw data for pipelines** — not optional in production
- Three broad task types: **reformatting, calculating, removing garbage**
- Traditional single-machine tools break down at scale; **Spark scales out**
- A Spark **schema** = `StructType` of `StructField(name, type, nullable)`
- Schemas give both **correctness** (filter non-conforming rows) and **performance** (skip inference → single-read)
- Types live in **`pyspark.sql.types`** — import `StructType`, `StructField`, and whichever leaf types the data needs

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("pyspark-cleaning").getOrCreate()

26/11/15 18:48:53 WARN Utils: Your hostname, DDG-1000 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/15 18:48:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/15 18:48:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/15 18:48:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
# Import `*` from the `pyspark.sql.types` library.
from pyspark.sql.types import *

# Define a new schema using the `StructType` method.
people_schema = StructType([
  # Define a `StructField` for `name`, `age`, and `city`. Each field should correspond to the correct datatype and not be `nullable`.
  StructField('name', StringType(), False),
  StructField('age', IntegerType(), False),
  StructField('city', StringType(), False)
])

## Immutability and Lazy Processing


- Two deeper Spark concepts that explain **how** DataFrame operations actually work under the hood
- Covers: variable mutability in general, **immutability** of Spark DataFrames, and **lazy processing**

### Variable review — mutability in most languages

In Python (and most other languages), variables are **fully mutable**:

- Their values can be changed at any time, as long as the variable is in scope
- Very flexible — but creates problems when **multiple concurrent components** try to modify the same data

Workarounds in traditional languages:

- **Mutexes**, **semaphores**, locks, and similar synchronisation primitives
- These add significant complexity, especially for non-trivial programs
- Tricky to get right; a common source of deadlocks and race conditions

### Immutability in Spark

Unlike typical Python variables, **Spark DataFrames are immutable** — a design choice drawn from **functional programming**.

What this means in practice:

- A DataFrame is **defined once** and **cannot be modified** after initialisation
- If a variable name is reused, the original data is **discarded** (unless still referenced elsewhere) and the name is reassigned to a new DataFrame
- **No in-place modification** — every "change" produces a new DataFrame

Why on earth is this efficient? Because immutability lets Spark **share data safely** across every component of the cluster — no need to worry about concurrent writes clobbering each other. The complexity of mutexes and locks simply goes away.

### Example — immutability in action

Load a CSV, add a column, drop another. The key thing to notice: the DataFrame isn't being *mutated* — each step produces a **new** DataFrame object that gets bound to the same variable name.

```python
# Step 1 — create a DataFrame from a CSV
voter_df = spark.read.csv('voterdata.csv')

# Step 2 — add a 'fullyear' column (2-digit year + 2000)
# This doesn't mutate voter_df. It creates a NEW DataFrame
# definition carrying the transformation and rebinds the name.
voter_df = voter_df.withColumn(
    'fullyear',
    voter_df['year'] + 2000
)

# Step 3 — drop the original 'year' column
# Again: new DataFrame, name rebound. Old one is destroyed.
voter_df = voter_df.drop('year')
```

Important to lock in:

- The **original** `year` column is **permanently gone from this DataFrame instance**
- The **underlying CSV file is untouched** — can always reload into a fresh DataFrame if the column is needed again
- Every "modification" is really a **new definition** being swapped into the variable name

### Lazy processing

So how does Spark do all this quickly, even on huge datasets? **Lazy processing**.

**Lazy processing** = very little actually happens until an **action** is performed.

In the previous example — reading a CSV, adding a column, dropping another:

- **No data is actually read, added, or modified at those points**
- Spark only **updates the instruction graph** (the chain of transformations) for what to do *when asked*

This gives Spark room to **optimise the plan** before executing anything — reordering, combining and pruning steps to produce the most efficient execution path.

Nothing runs until an **action** is called. Adding `count()` forces execution of every queued transformation:

```python
voter_df = spark.read.csv('voterdata.csv')

voter_df = voter_df.withColumn(
    'fullyear',
    voter_df['year'] + 2000
)

voter_df = voter_df.drop('year')

# This is an action — NOW Spark actually reads the file,
# applies both transformations, and returns a result.
voter_df.count()
```

Key point: transformations are **lazy**; actions are **eager**. The transformation chain is just a recipe — it doesn't cook anything until an action asks for the output.

**Common actions that force execution:**

- `count()`
- `show()`
- `collect()`
- `take(n)`
- `write(...)`

### Key takeaways — cheat sheet

| Concept | What it means | Why it matters |
|---|---|---|
| **Immutability** | DataFrames can't be modified in place; "changes" produce new DataFrames | Safe sharing across the cluster; no locking needed |
| **Lazy processing** | Transformations build a plan; nothing runs until an action | Lets Spark optimise the whole chain before executing |

**Things to lock in:**

- Spark DataFrames are **immutable** — every transformation returns a **new** DataFrame
- Reassigning to the same variable name **replaces** the old definition (original is garbage-collected if nothing else holds it)
- The **underlying data file is never touched** — only in-memory DataFrame definitions change
- **Transformations** (e.g. `withColumn`, `drop`, `filter`, `select`) are **lazy** — they just extend the plan
- **Actions** (e.g. `count`, `show`, `collect`, `write`) **trigger execution** of the plan
- Immutability + laziness together → safe concurrency **and** aggressive query optimisation; the two pillars of Spark's model

In [7]:
import pyspark.sql.functions as F

In [8]:
# Load the CSV file
aa_dfw_df = spark.read.format('csv') \
    .options(header=True) \
    .load('AA_DFW_2018.csv.gz')

# Add the airport column using the F.lower() method
aa_dfw_df = aa_dfw_df.withColumn(
    'airport',
    F.lower(aa_dfw_df['Destination Airport'])
)

# Show the DataFrame
aa_dfw_df.show()

+-----------------+-------------+-------------------+-----------------------------+-------+
|Date (MM/DD/YYYY)|Flight Number|Destination Airport|Actual elapsed time (Minutes)|airport|
+-----------------+-------------+-------------------+-----------------------------+-------+
|       01/01/2017|         0005|                HNL|                          537|    hnl|
|       01/01/2017|         0007|                OGG|                          498|    ogg|
|       01/01/2017|         0037|                SFO|                          241|    sfo|
|       01/01/2017|         0043|                DTW|                          134|    dtw|
|       01/01/2017|         0051|                STL|                           88|    stl|
|       01/01/2017|         0060|                MIA|                          149|    mia|
|       01/01/2017|         0071|                LAX|                          203|    lax|
|       01/01/2017|         0074|                MEM|                           

## Understanding Parquet


- Spark can read text and CSV files, but **CSV isn't always the best working format**
- Covers: CSV's limitations (general and Spark-specific), what **Parquet** is, how to read/write it, and using Parquet files with SQL

### Difficulties with CSV files

A few baked-in problems with the format itself:

- **No schema** — no data types, no column names beyond an optional header row
- **Delimiter collisions** — content containing a comma (or whatever the delimiter is) requires **escaping**
- **Escape collisions** — using the escape character inside content means yet **further escaping** (the escape-the-escape problem)
- **Limited encoding support** — available encodings depend on the language reading the file

### Spark-specific issues with CSV

On top of the general issues, CSV plays badly with Spark in particular:

- **Slow to import and parse** — text parsing is CPU-heavy
- **Cannot be shared between workers during import** — a single file can't be chunked across the cluster the way binary formats can
- **Schema must be inferred by reading all the data first** (if none is defined) — doubles the I/O cost
- **No predicate pushdown** — more on this in a moment

**Predicate pushdown**

A key Spark optimisation: **order tasks to do the least amount of work**.

- **Filter data *before* processing** → the main trick predicate pushdown applies
- Drastically reduces the amount of data that has to flow through the pipeline on large datasets
- **CSV cannot use predicate pushdown** — the whole file has to be read before anything is filtered

**Intermediate file representations**

Spark pipelines are often multi-step and benefit from caching intermediate results to disk. Using CSV for that means re-doing schema definition, encoding handling and parsing every single stage — a lot of wasted effort.

### The Parquet format

**Parquet** = a **compressed columnar** data format built for Hadoop-based systems (Spark, Hadoop, Apache Impala, …).

Key properties:

- **Columnar layout** → read only the columns actually needed
- **Compressed** → smaller files, faster I/O
- **Chunked structure** → efficient reads/writes **without processing the entire file**
- **Schema information embedded** in the file itself — no need to define it on every read
- **Encoding handled automatically** — no CSV-style escape headaches
- **Supports predicate pushdown** → significant performance improvements over CSV
- **Perfect as an intermediate / on-disk representation** for processed data

Important to lock in: Parquet is **binary**, so it can only be opened with proper tools — a text editor won't do. That's the one real cost compared to CSV, which anyone can crack open in Notepad.

### Working with Parquet

Reading is dead simple. Two equivalent styles:

```python
# Long form — using read.format
# .format() specifies the format; filename extension is ignored
df = spark.read.format('parquet').load('filename.parquet')

# Shortcut — generally preferred
df = spark.read.parquet('filename.parquet')
```

Writing mirrors reading:

```python
# Long form
df.write.format('parquet').save('filename.parquet')

# Shortcut
df.write.parquet('filename.parquet')
```

The **long form** is useful when extra option flags are needed — e.g. overwriting an existing file:

```python
df.write.format('parquet') \
    .mode('overwrite') \
    .save('filename.parquet')
```

Key point: the two forms are **interchangeable** — the shortcut is just syntactic sugar over `read.format('parquet')`.

### Parquet and SQL

Parquet pairs beautifully with Spark SQL — both because the schema is already defined and because predicate pushdown kicks in.

Workflow:

1. Read the Parquet file into a DataFrame
2. Register it as a temporary view with `createOrReplaceTempView()`
3. Query it with `spark.sql(...)`

```python
# 1. Read the Parquet file
flights_df = spark.read.parquet('flights.parquet')

# 2. Register as a temp view
flights_df.createOrReplaceTempView('flights')

# 3. Run a SQL query — e.g. flights under 100 minutes
short_flights_df = spark.sql(
    """
    SELECT *
    FROM flights
    WHERE flight_duration < 100
    """
)

short_flights_df.show()
```

Because the backing store is Parquet:

- **Schema is already defined** → no inference step
- **Predicate pushdown** → the `WHERE flight_duration < 100` filter is pushed down into the file read, so irrelevant chunks never leave disk

### Key takeaways — cheat sheet

**CSV vs Parquet at a glance:**

| | **CSV** | **Parquet** |
|---|---|---|
| Format | Text | Binary, columnar |
| Schema | None (or just a header) | Embedded |
| Compression | None by default | Built-in |
| Shareable between workers on import | No | Yes (chunked) |
| Predicate pushdown | No | **Yes** |
| Edit with a text editor | Yes | **No** |
| Good for intermediate storage | No | **Yes** |

**Parquet cheat sheet:**

```python
# Read
df = spark.read.parquet('file.parquet')

# Write
df.write.parquet('file.parquet')

# Overwrite an existing file
df.write.format('parquet').mode('overwrite').save('file.parquet')

# Query with SQL
df.createOrReplaceTempView('my_table')
spark.sql("SELECT * FROM my_table WHERE col > 100").show()
```

**Things to lock in:**

- CSV = fine for input/output, **poor for intermediate storage**
- Parquet = **columnar, compressed, schema-aware, predicate-pushdown friendly** → ideal for Spark workflows
- **Predicate pushdown** = filter early, in the file-read stage → huge win on big datasets
- Shortcut vs long-form read/write are **interchangeable**; long-form needed only when extra options (like `mode('overwrite')`) are required
- Parquet + `createOrReplaceTempView` + `spark.sql` = fast SQL analytics with minimal setup

In [23]:
df1 = spark.read.format("csv").load("AA_DFW_2017_Departures_Short.csv.gz")
df2 = spark.read.format("csv").load("AA_DFW_2016_Departures_Short.csv.gz")

In [24]:
# View the row count of df1 and df2
print("df1 Count: %d" % df1.count())
print("df2 Count: %d" % df2.count())

# Combine the DataFrames into one
df3 = df1.union(df2)

# Save the df3 DataFrame in Parquet format
df3.write.parquet('AA_DFW_ALL.parquet', mode='overwrite')

# Read the Parquet file into a new DataFrame and run a count
print(spark.read.parquet('AA_DFW_ALL.parquet').count())

df1 Count: 139359
df2 Count: 140605
279964


In [27]:
df3_renamed = df3.withColumnsRenamed(
    {
        "_c3": "flight_duration"
    }
)

df3_renamed.write.parquet('AA_DFW_ALL.parquet', mode='overwrite')

In [28]:
# Read the Parquet file into flights_df
flights_df = spark.read.parquet("AA_DFW_ALL.parquet")

# Register the temp table
flights_df.createOrReplaceTempView('flights')

# Run a SQL query of the average flight duration
avg_duration = spark.sql('SELECT avg(flight_duration) from flights').collect()[0]
print('The average flight time is: %d' % avg_duration)

The average flight time is: 151


In [22]:
flights_df.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)



## DataFrame Column Operations

- Focus: using **Spark column operations** to clean data
- Covers: DataFrame refresher, where operations live (`F` vs column methods vs DataFrame methods), common transformations, filtering, string manipulation, and `ArrayType()` columns

### DataFrame refresher

Quick recap on what a Spark DataFrame actually is:

- Made up of **rows and columns** — generally analogous to a database table
- **Immutable** — any change to the structure or content creates a **new DataFrame**
- Modified only through **transformations** (never in place)

Two illustrative transformations:

```python
# .filter() — keep only rows matching a condition
df.filter(df.name.startswith('M'))

# .select() — keep only specified columns
df.select('name', 'position')
```

### Where operations live — three homes

Spark operations split into **three families**, each with a different home. Knowing which lives where kills most of the guessing about syntax.

| Home | What lives there | Mental test |
|---|---|---|
| **`F.something(col)`** | Transformations that produce a **value** from a column | "Would this be a SQL function? `UPPER(name)`, `YEAR(date)`?" |
| **`col.something()`** | Questions/actions about a **specific column** | "Is this about *this column's* type, nullability, naming, or a per-value test?" |
| **`df.something()`** | Operations on the whole **DataFrame** | "Is this about rows, joins, grouping, or sorting?" |

**The `F` camp — SQL built-in functions**

Conventionally imported as `F`:

```python
import pyspark.sql.functions as F
```

Anything that in SQL would look like `FUNCTION(col)` lives here:

```python
F.upper(col)        F.lower(col)        F.length(col)
F.year(col)         F.month(col)        F.to_date(col)
F.split(col, ' ')   F.size(arr_col)
F.round(col, 2)     F.abs(col)
F.concat(c1, c2)    F.regexp_replace(col, r'\d+', '')
F.when(cond, a).otherwise(b)
F.coalesce(c1, c2)
F.lit(42)           # wrap a literal as a Column
F.col('name')       # grab a column by name

# Aggregates live here too
F.sum(col)   F.avg(col)   F.count(col)   F.max(col)   F.min(col)
```

**The column-method camp — metadata, type, per-value tests**

Called directly on the column object (`df.col_name.something()`):

```python
# Type conversion
df.age.cast(IntegerType())

# Renaming
df.name.alias('full_name')

# Null tests
df.name.isNotNull()
df.name.isNull()

# Per-value string tests
df.name.contains('Smith')
df.name.startswith('M')
df.name.endswith('son')
df.name.like('%Smith%')         # SQL LIKE
df.name.rlike(r'^M.*')           # regex LIKE

# Membership / ranges
df.age.isin([18, 21, 65])
df.age.between(18, 65)

# Array / map access
df.splits.getItem(0)
```

Plus **operators** — always inline, never via `F`:

```python
df.age + 10            df.price * 1.2
df.age > 18            df.name == 'Alice'
(df.age > 18) & (df.country == 'UK')
~df.name.contains('Smith')
```

**The DataFrame camp**

```python
df.select(...)    df.filter(...)    df.where(...)
df.withColumn(name, expr)           df.drop(...)
df.groupBy(...).agg(...)            df.orderBy(...)
df.join(other, ...)
df.show()         df.count()        df.collect()
```

**Quick heuristics for picking the right home:**

1. **SQL built-in?** → `F.*`. If writing `UPPER(x)`, `LENGTH(x)`, `YEAR(x)` in SQL, it's in `F`.
2. **About *this* column's identity/type/nullability?** → column method (`cast`, `alias`, `isNull`, `isin`, `between`, `desc`, `asc`).
3. **Per-row string test?** → column method (`contains`, `startswith`, `like`) — **not** `F`. This is the trap most people fall into.
4. **Operator (`+`, `>`, `&`, `~`)?** → inline; never `F`.
5. **Changes the shape of the data (rows/columns)?** → DataFrame method.

Important to lock in: when in doubt, **IDE autocomplete** is the fastest tiebreaker — type `df.col.` and see what methods appear; type `F.` and see what functions appear.

### Common DataFrame transformations

The ones that come up again and again:

**`.filter()`** — keep only rows satisfying the condition (analogous to SQL's `WHERE`). Spark provides `.where()` as a straight alias — use whichever reads better.

```python
# Only rows where vote_date is after 1 Jan 2019
df.filter(df.vote_date > '2019-01-01')

# Equivalent
df.where(df.vote_date > '2019-01-01')
```

**`.select()`** — return only the requested columns.

```python
df.select('name', 'position', 'vote_date')
```

**`.withColumn()`** — create a **new column**. First argument is the name; second is the expression that produces its values.

```python
import pyspark.sql.functions as F

# Add a 'year' column extracted from the vote_date column
df = df.withColumn('year', F.year(df.vote_date))
```

**`.drop()`** — remove a column from the DataFrame.

```python
df = df.drop('old_column')
```

Important to lock in: every one of these returns a **new DataFrame** — the original is unchanged (immutability from the previous chapter).

### Filtering data

Among the most common cleaning operations. `.filter()` is a workhorse for:

- **Removing null values**
- **Removing odd entries** — anything that doesn't fit the expected format
- **Splitting a DataFrame** containing combined data (e.g. one syslog file carrying multiple types of record)

Useful idioms:

```python
import pyspark.sql.functions as F

# Remove nulls in a specific column
df.filter(df.name.isNotNull())

# Keep only rows where 'name' contains 'Smith'
df.filter(df.name.contains('Smith'))

# Negate with ~ — exclude rows that contain 'Smith'
df.filter(~df.name.contains('Smith'))
```

Key point: the **tilde `~`** is the boolean NOT operator in PySpark expressions — works on any boolean column/condition, not just `.contains()`.

### Column string transformations

Modifying and converting strings is a big part of data cleaning. Most of the functions live in `pyspark.sql.functions`:

**Upper / lower casing (both via `F`):**

```python
# Create an 'upper' column with uppercase versions of 'name'
df = df.withColumn('upper', F.upper(df.name))

# Or lowercase
df = df.withColumn('lower', F.lower(df.name))
```

**Splitting strings → ArrayType (via `F`):**

```python
# Split 'name' on spaces → returns a column of lists (ArrayType)
df = df.withColumn('splits', F.split(df.name, ' '))
```

This is an example of an **intermediary column** — a column created purely as a stepping stone for further processing. Useful for keeping complex multi-step transformations readable.

**Type casting with `.cast()` (column method — not `F`):**

```python
from pyspark.sql.types import IntegerType

# Convert a string column to IntegerType
df = df.withColumn('age_int', df.age.cast(IntegerType()))
```

Important to remember: `cast()` is called on the **column object**, not on `F`. Common targets: `IntegerType()`, `FloatType()`, `DateType()`, etc.

### `ArrayType()` column functions

`ArrayType()` columns are the Spark equivalent of Python lists — a column where each row holds a list of values (often the result of `F.split(...)`).

Two go-to functions — **one in `F`, one a column method**:

**`F.size()`** — number of elements in the array column:

```python
# How many words after splitting on space
df = df.withColumn('word_count', F.size(df.splits))
```

**`.getItem(index)`** — grab the element at a specific index (0-based):

```python
# First word → 'first_name'
df = df.withColumn('first_name', df.splits.getItem(0))

# Second word → 'last_name'
df = df.withColumn('last_name',  df.splits.getItem(1))
```

Important to lock in: `getItem` is called on the **column itself**, not via `F`. Classic example of the F-vs-method split.

Spark has **many more** transformations and utility functions — worth consulting the official docs when building production pipelines.

### Key takeaways — cheat sheet

**The three homes at a glance:**

| Family | Pattern | Examples |
|---|---|---|
| **`F.*`** | `F.function(col)` | `F.upper`, `F.year`, `F.split`, `F.size`, `F.when`, `F.sum`, `F.lit` |
| **Column methods** | `col.method()` | `.cast`, `.alias`, `.isNull`, `.contains`, `.like`, `.isin`, `.between`, `.getItem`, `.desc` |
| **DataFrame methods** | `df.method()` | `.filter`, `.select`, `.withColumn`, `.drop`, `.groupBy`, `.orderBy`, `.join` |

**Transformations most used in cleaning:**

| Operation | What it does | Example |
|---|---|---|
| `.filter(cond)` / `.where(cond)` | Keep rows matching condition | `df.filter(df.age > 18)` |
| `.select(cols)` | Keep specified columns | `df.select('name', 'age')` |
| `.withColumn(name, expr)` | Add or replace a column | `df.withColumn('upper', F.upper(df.name))` |
| `.drop(col)` | Remove a column | `df.drop('temp_col')` |
| `.cast(type)` | Convert a column's type | `df.age.cast(IntegerType())` |

**String-manipulation one-liners (with `import pyspark.sql.functions as F`):**

```python
F.upper(col)           # UPPERCASE
F.lower(col)           # lowercase
F.split(col, sep)      # String → ArrayType
F.size(array_col)      # Length of an ArrayType column
array_col.getItem(i)   # i-th element of an ArrayType column
col.cast(IntegerType()) # type conversion
```

**Things to lock in:**

- DataFrames are **immutable** — reassign the result, don't expect in-place changes
- Conventional import: **`import pyspark.sql.functions as F`**
- **`F.*`** = SQL-style value transformations; **column methods** = questions/metadata about a column; **DataFrame methods** = shape-changing operations
- `.filter()` and `.where()` are **identical** — use whichever reads better
- **`~`** negates a boolean column/condition
- `.cast(...)` and `.getItem(i)` live on the **column**, not on `F` — classic gotchas
- Operators (`+`, `>`, `&`, `~`) are always inline, never via `F`
- When stuck on exact function names, reach for **IDE autocomplete** or the **PySpark SQL functions reference**

In [30]:
voter_df = spark.read.csv("DallasCouncilVoters.csv.gz", header = True)

voter_df.show()

+----------+-------------+-------------------+
|      DATE|        TITLE|         VOTER_NAME|
+----------+-------------+-------------------+
|02/08/2017|Councilmember|  Jennifer S. Gates|
|02/08/2017|Councilmember| Philip T. Kingston|
|02/08/2017|        Mayor|Michael S. Rawlings|
|02/08/2017|Councilmember|       Adam Medrano|
|02/08/2017|Councilmember|       Casey Thomas|
|02/08/2017|Councilmember|Carolyn King Arnold|
|02/08/2017|Councilmember|       Scott Griggs|
|02/08/2017|Councilmember|   B. Adam  McGough|
|02/08/2017|Councilmember|       Lee Kleinman|
|02/08/2017|Councilmember|      Sandy Greyson|
|02/08/2017|Councilmember|  Jennifer S. Gates|
|02/08/2017|Councilmember| Philip T. Kingston|
|02/08/2017|        Mayor|Michael S. Rawlings|
|02/08/2017|Councilmember|       Adam Medrano|
|02/08/2017|Councilmember|       Casey Thomas|
|02/08/2017|Councilmember|Carolyn King Arnold|
|02/08/2017|Councilmember| Rickey D. Callahan|
|01/11/2017|Councilmember|  Jennifer S. Gates|
|04/25/2018|C

In [31]:
# Show the distinct VOTER_NAME entries
voter_df.select('VOTER_NAME').distinct().show(40, truncate=False)

# Filter voter_df where the VOTER_NAME is 1-20 characters in length
voter_df = voter_df.filter('length(VOTER_NAME) > 0 and length(VOTER_NAME) < 20')

# Filter out voter_df where the VOTER_NAME contains an underscore
voter_df = voter_df.filter(~ F.col('VOTER_NAME').contains("_"))

# Show the distinct VOTER_NAME entries again
voter_df.select('VOTER_NAME').distinct().show(40, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|VOTER_NAME                                                                                                                                                                                                                                                                                                                                                                                                                 |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [35]:
# Add a new column called splits separated on whitespace
voter_df = voter_df.withColumn("splits", F.split(voter_df.VOTER_NAME, '\s+'))

# Create a new column called first_name based on the first item in splits
voter_df = voter_df.withColumn("first_name", voter_df.splits.getItem(0))

# Get the last entry of the splits list and create a column called last_name
voter_df = voter_df.withColumn("last_name", voter_df.splits.getItem(F.size('splits') - 1))

# Drop the splits column
voter_df = voter_df.drop('splits')

# Show the voter_df DataFrame
voter_df.show()

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_30353/4199570017.py:2: SyntaxWarning: invalid escape sequence '\s'
  voter_df = voter_df.withColumn("splits", F.split(voter_df.VOTER_NAME, '\s+'))
/home/mothma/dc-pyspark/.venv/lib/python3.12/site-packages/pyspark/sql/column.py:460: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


+----------+-------------+-------------------+----------+---------+
|      DATE|        TITLE|         VOTER_NAME|first_name|last_name|
+----------+-------------+-------------------+----------+---------+
|02/08/2017|Councilmember|  Jennifer S. Gates|  Jennifer|    Gates|
|02/08/2017|Councilmember| Philip T. Kingston|    Philip| Kingston|
|02/08/2017|        Mayor|Michael S. Rawlings|   Michael| Rawlings|
|02/08/2017|Councilmember|       Adam Medrano|      Adam|  Medrano|
|02/08/2017|Councilmember|       Casey Thomas|     Casey|   Thomas|
|02/08/2017|Councilmember|Carolyn King Arnold|   Carolyn|   Arnold|
|02/08/2017|Councilmember|       Scott Griggs|     Scott|   Griggs|
|02/08/2017|Councilmember|   B. Adam  McGough|        B.|  McGough|
|02/08/2017|Councilmember|       Lee Kleinman|       Lee| Kleinman|
|02/08/2017|Councilmember|      Sandy Greyson|     Sandy|  Greyson|
|02/08/2017|Councilmember|  Jennifer S. Gates|  Jennifer|    Gates|
|02/08/2017|Councilmember| Philip T. Kingston|  

## Conditional DataFrame Column Operations


- Moves beyond **blanket transformations** to transformations that depend on the **content of each row**
- Covers: Spark's built-in conditional clauses **`.when()`** and **`.otherwise()`**, plus why to use them over hand-rolled Python if/else

### Conditional clauses — the idea

Everything covered so far has been a **blanket transformation** — applied to every row the same way, regardless of data.

Often what's actually needed is: "do X if the row looks like Y, else do Z". Traditional if / then / else, but **per row**.

Spark provides built-in conditional clauses for exactly this, and they're preferable to writing a Python if/else that runs per row:

- **Hand-rolled if/else** → each row evaluated independently in the Python interpreter → **serious performance degradation**, especially on large datasets (the UDF overhead kills the pipeline)
- **Built-in `.when()` / `.otherwise()`** → runs inside Spark's **optimised execution engine** → columnar, pushed down, catalyst-optimised

Two components to the conditional clauses:

- **`.when()`** — the "if ... then" part
- **`.otherwise()`** — the optional "else" part

### `.when()` — the if/then

Lives in `pyspark.sql.functions`. Takes **two arguments**:

1. The **condition** to check
2. The **value** to return if the condition is `True`

```python
import pyspark.sql.functions as F

# DataFrame with Name and Age columns
df.select(
    df.Name,
    df.Age,
    F.when(df.Age >= 18, 'Adult')
).show()
```

Result: each row with `Age >= 18` gets an extra column with `'Adult'`; otherwise the new column is **null** (nothing matched, no `.otherwise()` given).

Key points:

- The third argument to `.select()` is a **brand-new column** generated on the fly — no `.withColumn()` needed
- The new column is **unnamed by default** — Spark auto-labels it something like `CASE WHEN ... END`
- If the `.when()` condition fails and there's no `.otherwise()`, the cell is **null**

**Naming the generated column:**

Add `.alias()` to give it a sensible header:

```python
df.select(
    df.Name,
    df.Age,
    F.when(df.Age >= 18, 'Adult').alias('AgeGroup')
).show()
```

### Chaining `.when()` — the "else if" pattern

Multiple `.when()` calls can be chained for an if / else-if / else-if structure:

```python
df.select(
    df.Name,
    df.Age,
    F.when(df.Age >= 18, 'Adult')
     .when(df.Age <  18, 'Minor')
     .alias('AgeGroup')
).show()
```

Important to lock in:

- `.when()` can be chained **as many times as needed**
- Each `.when()` is evaluated **in order** — first match wins
- Rows matching **no** `.when()` clause still return **null** unless followed by `.otherwise()`

### `.otherwise()` — the else

Analogous to the `else` branch in traditional if/else. Takes a **single argument**: what to return when none of the preceding `.when()` clauses matched.

```python
df.select(
    df.Name,
    df.Age,
    F.when(df.Age >= 18, 'Adult')
     .otherwise('Minor')
     .alias('AgeGroup')
).show()
```

Result: every row gets either `'Adult'` or `'Minor'` — no nulls.

Important to remember:

- **Many `.when()` clauses** can be chained → but there can be **only one `.otherwise()`** per chain
- `.otherwise()` sits at the end — it's the fallback for everything that didn't match

Compare the two equivalent approaches:

```python
# With otherwise()
F.when(df.Age >= 18, 'Adult').otherwise('Minor')

# Without otherwise() — same result because the second .when() covers the rest
F.when(df.Age >= 18, 'Adult').when(df.Age < 18, 'Minor')
```

Both produce the same DataFrame. The `.otherwise()` form is usually cleaner when there's a clear default.

### Key takeaways — cheat sheet

**Pattern template:**

```python
import pyspark.sql.functions as F

df.withColumn(
    'new_col',
    F.when(cond1, value1)
     .when(cond2, value2)
     .otherwise(default_value)
)
```

**Side-by-side with traditional if/else:**

| Traditional code | Spark equivalent |
|---|---|
| `if cond: return x` | `F.when(cond, x)` |
| `elif cond2: return y` | `.when(cond2, y)` |
| `else: return z` | `.otherwise(z)` |

**Things to lock in:**

- Use **`F.when()` / `.otherwise()`** instead of hand-rolled Python if/else on each row — vastly better performance
- `.when()` takes **(condition, value)**; `.otherwise()` takes just a **value**
- Chain as many `.when()` clauses as needed; **only one `.otherwise()`** at the end
- First match wins — order the clauses from most-specific to most-general
- A row that matches **no** `.when()` and has no `.otherwise()` → **null**
- The generated column is unnamed by default — add **`.alias('name')`** or wrap in `.withColumn('name', ...)` to label it
- `.when()` lives in **`pyspark.sql.functions`** (the `F` family — a value-producing SQL-style function)

In [38]:
# Add a column to voter_df named random_val with the results of the F.rand() method for any voter with the title Councilmember
voter_df = voter_df.withColumn('random_val',
                               F.when(voter_df.TITLE == "Councilmember", F.rand()))

# Show some of the DataFrame rows, noting whether the when clause worked
voter_df.show()

+----------+-------------+-------------------+----------+---------+--------------------+
|      DATE|        TITLE|         VOTER_NAME|first_name|last_name|          random_val|
+----------+-------------+-------------------+----------+---------+--------------------+
|02/08/2017|Councilmember|  Jennifer S. Gates|  Jennifer|    Gates| 0.17006448480485847|
|02/08/2017|Councilmember| Philip T. Kingston|    Philip| Kingston|  0.8375575240572303|
|02/08/2017|        Mayor|Michael S. Rawlings|   Michael| Rawlings|                NULL|
|02/08/2017|Councilmember|       Adam Medrano|      Adam|  Medrano| 0.15432910472738404|
|02/08/2017|Councilmember|       Casey Thomas|     Casey|   Thomas|  0.6283395064524638|
|02/08/2017|Councilmember|Carolyn King Arnold|   Carolyn|   Arnold|  0.9976728099739987|
|02/08/2017|Councilmember|       Scott Griggs|     Scott|   Griggs| 0.37234128676123723|
|02/08/2017|Councilmember|   B. Adam  McGough|        B.|  McGough|  0.8931703590025484|
|02/08/2017|Councilme

In [49]:
# Add a column to `voter_df` named `random_val` with the results of the `F.rand()` method for any voter with the title Councilmember. Set `random_val` to 2 for the Mayor. Set any other title to the value 0.
voter_df = voter_df.withColumn('random_val',
                               F.when(voter_df.TITLE == 'Councilmember', F.rand())
                               .when(voter_df.TITLE == 'Mayor', 2)
                               .otherwise(0)
)

# Show some of the Data Frame rows, noting whether the clauses worked.
voter_df.show()

# Use the `.filter` clause to find 0 in `random_val`.
voter_df.filter(voter_df.random_val ==0).show()

+----------+-------------+-------------------+----------+---------+-------------------+
|      DATE|        TITLE|         VOTER_NAME|first_name|last_name|         random_val|
+----------+-------------+-------------------+----------+---------+-------------------+
|02/08/2017|Councilmember|  Jennifer S. Gates|  Jennifer|    Gates|0.04698799689169286|
|02/08/2017|Councilmember| Philip T. Kingston|    Philip| Kingston| 0.3828731718520767|
|02/08/2017|        Mayor|Michael S. Rawlings|   Michael| Rawlings|                2.0|
|02/08/2017|Councilmember|       Adam Medrano|      Adam|  Medrano| 0.2782261415573011|
|02/08/2017|Councilmember|       Casey Thomas|     Casey|   Thomas|0.34476340989288845|
|02/08/2017|Councilmember|Carolyn King Arnold|   Carolyn|   Arnold| 0.7035311439032808|
|02/08/2017|Councilmember|       Scott Griggs|     Scott|   Griggs|0.24932916120948223|
|02/08/2017|Councilmember|   B. Adam  McGough|        B.|  McGough|  0.498317020029724|
|02/08/2017|Councilmember|      

## User Defined Functions

- Built-in Spark functions cover most cases — but sometimes **custom logic** is needed
- A **UDF** = a Python function wrapped so Spark can apply it to a column
- Covers: how to define, how to register with `F.udf()`, the `@udf` decorator shortcut, how to call, plus the performance caveat worth knowing up front

### What is a UDF?

**UDF** (User Defined Function) = a plain Python method the user writes to perform some specific bit of logic that no built-in Spark function covers.

**The workflow:**

1. Write a normal Python function
2. Wrap it with `pyspark.sql.functions.udf()`, telling Spark what **return type** to expect
3. Store the wrapped version as a variable
4. Use it like any other Spark function (inside `withColumn`, `select`, etc.)

Under the hood, Spark calls the Python function **once per row**, passing the value of the specified column(s) in and receiving the return value back.

### Example — a `reverseString` UDF

Trivial-but-illustrative example: reverse the characters of a string.

**Step 1 — define the Python function:**

```python
def reverseString(mystr):
    return mystr[::-1]   # Python slice shorthand for reverse
```

Nothing Spark-specific — just regular Python. Given `"help"`, returns `"pleh"`.

**Step 2 — wrap it with `F.udf()`:**

```python
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

udfReverseString = F.udf(reverseString, StringType())
```

Two arguments to `F.udf()`:

- The **Python function** itself
- The **Spark return type** (any type from `pyspark.sql.types` — can even be a complex `ArrayType` or a full schema)

Typical return types: `StringType()`, `IntegerType()`, `FloatType()`, `ArrayType(...)`.

**Step 3 — apply it to a DataFrame:**

```python
user_df = user_df.withColumn(
    'ReverseName',
    udfReverseString(user_df.Name)
)
```

The column `user_df.Name` is passed to the UDF; Spark calls the Python function row-by-row and stashes the results in the new `ReverseName` column.

### Syntactic sugar — the `@udf` decorator

Python decorators give a cleaner one-step way to write the exact same thing:

```python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(StringType())
def reverseString(mystr):
    return mystr[::-1]

user_df = user_df.withColumn('ReverseName', reverseString(user_df.Name))
```

**How it relates to the function-call form:**

Decorators are just sugar — these two are **exactly equivalent** under the hood:

```python
@udf(StringType())
def reverseString(mystr):
    return mystr[::-1]

# ↑↑↑ expands to ↓↓↓

def reverseString(mystr):
    return mystr[::-1]
reverseString = udf(StringType())(reverseString)
```

Same wrapping, same behaviour, **same performance characteristics**. Pick whichever reads better.

**Decorator shortcuts worth knowing:**

```python
@udf                     # no parens — defaults return type to StringType()
def greet(name):
    return f"Hello, {name}"

@udf("int")              # Spark SQL type string also works
def length(s):
    return len(s)

@udf("array<string>")    # complex type via SQL string
def split_comma(s):
    return s.split(",")
```

**When to reach for which:**

| Situation | Favour |
|---|---|
| Python function also used *outside* Spark (tests, other code paths) | `F.udf(fn, type)` — keeps original `fn` name intact |
| Python function exists **only** to be called through Spark | `@udf(type)` — one block, type sits next to definition |
| Teaching material / clarity for a beginner audience | `F.udf(fn, type)` — more explicit about what's happening |

Important to remember: the decorator doesn't buy any speed — the Python UDF overhead applies equally to both forms. It's **purely a syntax preference**.

### Argument-less UDFs

UDFs don't have to take a column argument — they can produce a value on their own.

Example: pick a random "house" for each row.

```python
import random
from pyspark.sql.types import StringType

def sortingCap():
    return random.choice(['G', 'H', 'R', 'S'])

udfSortingCap = F.udf(sortingCap, StringType())

user_df = user_df.withColumn('House', udfSortingCap())
```

Equivalent with the decorator:

```python
@udf(StringType())
def sortingCap():
    return random.choice(['G', 'H', 'R', 'S'])

user_df = user_df.withColumn('House', sortingCap())
```

Key differences from the previous example:

- No column passed in — just `udfSortingCap()` / `sortingCap()` with empty parentheses
- Still called **once per row**, so each row gets its own random letter

### Important caveat — performance

UDFs are a **big escape hatch**, but they come at a real cost. Important to lock in:

- The Python function runs in a separate **Python interpreter** process on each executor
- For every row, Spark has to serialise the value **from JVM to Python**, run the function, then serialise the result **back from Python to JVM**
- **Catalyst cannot optimise inside the UDF** — it's a black box to the query planner
- No **vectorisation**, no **predicate pushdown** through the UDF

So the rule of thumb:

- If a **built-in function** from `F` can do it → **use that**, every time
- If `F.when() / .otherwise()` can express the logic → **use that**
- Reach for a UDF only when **no combination of built-ins works**

For serious performance, Spark also offers **pandas UDFs** (vectorised UDFs using Apache Arrow), which batch many rows into a pandas Series and process them in one Python call — orders of magnitude faster than row-at-a-time UDFs. Worth knowing about for later.

### Key takeaways — cheat sheet

**Two equivalent UDF templates:**

```python
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# Function-call form
def my_fn(x):
    return ...

udf_my_fn = F.udf(my_fn, StringType())
df = df.withColumn('new_col', udf_my_fn(df.some_col))

# Decorator form — syntactic sugar for the above
@F.udf(StringType())
def my_fn(x):
    return ...

df = df.withColumn('new_col', my_fn(df.some_col))
```

**Things to lock in:**

- **UDF** = plain Python function + `F.udf(fn, return_type)` wrapper + applied like any Spark function
- Second argument to `F.udf()` is the **return type** (from `pyspark.sql.types`)
- **`@udf(ReturnType())`** is pure syntactic sugar for the same wrapping — no speed difference, just prettier
- `@udf` with no parens defaults to **`StringType()`** — convenient, but always better to specify the type explicitly
- Spark SQL type strings (`"int"`, `"string"`, `"array<string>"`) work in place of the typed classes
- Called **once per row** — Python runs for every row
- Argument-less UDFs are fine — call with empty parens
- **Performance cost is real** — JVM↔Python serialisation per row, no Catalyst optimisation
- Prefer **built-in `F` functions** whenever they can do the job
- For heavy workloads, **pandas UDFs** (vectorised) are the better escape hatch

In [53]:
voter_df = voter_df.withColumn("splits", F.split(voter_df.VOTER_NAME, '\s+'))

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_30353/1354151728.py:1: SyntaxWarning: invalid escape sequence '\s'
  voter_df = voter_df.withColumn("splits", F.split(voter_df.VOTER_NAME, '\s+'))


In [55]:
def getFirstAndMiddle(names):
  # Edit the `getFirstAndMiddle()` function to return a space separated string of names, except the last entry in the names list.
  return ' '.join(names[:-1])

# Define the function as a user-defined function. It should return a string type.
udfFirstAndMiddle = F.udf(getFirstAndMiddle, StringType())

# Create a new column on `voter_df` called `first_and_middle_name` using your UDF.
voter_df = voter_df.withColumn('first_and_middle_name', udfFirstAndMiddle(voter_df.splits))

# Show the Data Frame.
voter_df.show()

+----------+-------------+-------------------+----------+---------+-------------------+---------------------+--------------------+
|      DATE|        TITLE|         VOTER_NAME|first_name|last_name|         random_val|first_and_middle_name|              splits|
+----------+-------------+-------------------+----------+---------+-------------------+---------------------+--------------------+
|02/08/2017|Councilmember|  Jennifer S. Gates|  Jennifer|    Gates|0.04698799689169286|          Jennifer S.|[Jennifer, S., Ga...|
|02/08/2017|Councilmember| Philip T. Kingston|    Philip| Kingston| 0.3828731718520767|            Philip T.|[Philip, T., King...|
|02/08/2017|        Mayor|Michael S. Rawlings|   Michael| Rawlings|                2.0|           Michael S.|[Michael, S., Raw...|
|02/08/2017|Councilmember|       Adam Medrano|      Adam|  Medrano| 0.2782261415573011|                 Adam|     [Adam, Medrano]|
|02/08/2017|Councilmember|       Casey Thomas|     Casey|   Thomas|0.34476340989288

## Partitioning and Lazy Processing

- Steps back from transformations to look at **how Spark actually processes data** under the hood
- Covers: **partitioning**, **lazy processing** (recap with a new angle), the problem of adding **IDs** in a distributed world, and `monotonically_increasing_id()`

### Partitioning

Spark breaks DataFrames into **partitions** — chunks of data distributed across the cluster.

- Partitions can be automatically **defined, enlarged, or shrunk**
- Count and size vary with the cluster type and configuration
- **Rule of thumb**: try to keep partition sizes roughly equal (skewed partitions kill parallelism — one slow worker holds up everything)

**Why partitions matter for performance:**

Each partition is handled **independently** by whichever node picks it up. That independence is the key to Spark's horizontal scaling:

- Workers don't need to **compete for shared resources**
- Workers don't need to **consult other nodes** for answers
- Each worker can schedule its own work optimally

Important to lock in: this independence is *why* Spark scales. Any operation that forces workers to coordinate (shuffles, global sequences, etc.) is expensive precisely because it breaks the independence.

### Lazy processing (revisited)

Previously covered — quick refresher with a new angle:

- Any **transformation** is lazy — it's more like a **recipe** than a command
- Transformations: `.withColumn()`, `.select()`, `.filter()`, `.drop()`, etc. → they **define** what to do, don't actually do it
- **Actions** (`.count()`, `.show()`, `.write()`, `.collect()`) trigger execution

The new wrinkle worth noting:

- **Spark can reorder transformations** for best performance
- Usually invisible — results are the same
- **Occasionally causes unexpected behaviour** — e.g. IDs not being assigned until after other transformations run
- Doesn't produce *wrong* answers, but the data can look **unusual** if the reorder isn't anticipated

This is the kind of thing that catches people out when they add `monotonically_increasing_id()` expecting sequential numbers and then wonder why joins look odd.

### Adding IDs — the problem

Relational databases typically have an **ID column** — an integer, sequential, unique — used to identify rows (foreign keys, joins, deduplication).

**Why traditional sequential IDs don't fit Spark:**

- Each new ID = previous + 1
- Multiple workers would all need to consult a **shared counter** for "what's next"
- Fine on a single server
- A **serious bottleneck** on a distributed platform — forces coordination between workers that otherwise wouldn't talk to each other

So Spark can't just generate sequential IDs the way Postgres does. A different design is needed.

### `monotonically_increasing_id()`

Spark's built-in solution. Produces an ID that is:

- **An integer**
- **Increasing in value**
- **Unique**
- **Parallel-friendly** — each partition gets its own range, no coordination needed

**Crucial catch:** the IDs are **not sequential**. There can be large gaps between values.

```python
import pyspark.sql.functions as F

df = df.withColumn('row_id', F.monotonically_increasing_id())
df.show()
```

Sample output (illustrative):

```
+------+----+-------------+
|  name| age|       row_id|
+------+----+-------------+
|Alice |  30|            0|
|Bob   |  25|            1|
|Carol |  40|  8589934592 |
|David |  35|  8589934593 |
+------+----+-------------+
```

Notice: integers, increasing, unique — but **not consecutive**. The jump between rows 2 and 3 is where a new partition started.

**How it works under the hood:**

- The ID is a **64-bit integer**, effectively split into groups per partition
- Each partition gets up to **~8.4 billion IDs** to assign within its own range
- There are **~2.1 billion possible partitions**, and the ranges **never overlap**
- That's how uniqueness holds without any inter-worker coordination

Key insight: the ID's **top bits encode the partition number**; the **bottom bits** are the sequential position within that partition. Because each partition owns a disjoint range, no worker ever has to ask another one what to use next.

### Practical notes

A few practical points worth remembering:

- Partitioning and `monotonically_increasing_id()` have **a lot of nuance** — the above is the shape, not the whole story
- Spark being **lazy** is often the key to troubleshooting weird behaviour — if something looks out of order, remember: the transformations may have been **reordered** before execution
- **Operations can be out of order**, especially when **joins** are involved — a join can re-partition data, scrambling what felt like a stable ordering
- Always **test transformations** on small data before trusting assumptions about order

### Key takeaways — cheat sheet

**Partition + ID model:**

| Concept | Key point |
|---|---|
| **Partition** | Independent chunk of a DataFrame; workers process partitions without coordinating |
| **Lazy transformations** | Can be reordered by Spark's optimiser before execution |
| **Traditional sequential IDs** | Require shared state → bottleneck for distributed systems |
| **`monotonically_increasing_id()`** | Unique, increasing, **not sequential**; each partition owns its own range |

**ID generation template:**

```python
import pyspark.sql.functions as F

df_with_id = df.withColumn('row_id', F.monotonically_increasing_id())
```

**Things to lock in:**

- Spark's parallelism comes from **partitions acting independently** — any operation forcing coordination costs performance
- Keep partition sizes roughly **balanced** when possible — skew is a silent killer
- Transformations are **lazy and reorderable**; don't assume execution order from code order
- `monotonically_increasing_id()` gives **unique, increasing** IDs — **not consecutive**
- Each partition owns its own ID range (~8.4 billion values), so no worker coordination needed → no bottleneck
- If joins or other partition-shuffling operations are in the chain, IDs may appear in unexpected places — **test on small data first**

In [59]:
df = spark.read.format("csv").options(header=True).load("DallasCouncilVotes.csv.gz")

df.printSchema()

root
 |-- DATE: string (nullable = true)
 |-- AGENDA_ITEM_NUMBER: string (nullable = true)
 |-- ITEM_TYPE: string (nullable = true)
 |-- DISTRICT: string (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- VOTER NAME: string (nullable = true)
 |-- VOTE CAST: string (nullable = true)
 |-- FINAL ACTION TAKEN: string (nullable = true)
 |-- AGENDA ITEM DESCRIPTION: string (nullable = true)
 |-- AGENDA_ID: string (nullable = true)
 |-- VOTE_ID: string (nullable = true)



In [ ]:
# Select the unique entries from the column `VOTER NAME` and create a new DataFrame called `voter_df`.
voter_df = df.select(df["VOTER NAME"]).distinct()

# # Count the rows in the `voter_df` DataFrame.
print("\nThere are %d rows in the voter_df DataFrame.\n" % voter_df.count())

# # Add a ROW_ID column using the appropriate Spark function.
voter_df = voter_df.withColumn('ROW_ID', F.monotonically_increasing_id())

# Show the rows with the 10 highest ROW_IDs.
voter_df.orderBy(voter_df.ROW_ID.desc()).show(10)


There are 36 rows in the voter_df DataFrame.

+--------------------+------+
|          VOTER NAME|ROW_ID|
+--------------------+------+
|                NULL|    35|
|        Lee Kleinman|    34|
|  the  final  201...|    33|
|         Erik Wilson|    32|
|  the  final   20...|    31|
| Carolyn King Arnold|    30|
| Rickey D.  Callahan|    29|
|   the   final  2...|    28|
|    Monica R. Alonzo|    27|
|     Lee M. Kleinman|    26|
+--------------------+------+
only showing top 10 rows



In [66]:
voter_df_single = voter_df.select(F.col("VOTER NAME")).repartition(1)
voter_df = voter_df.select(F.col("VOTER NAME")).repartition(2)

In [69]:
# Print the number of partitions in each DataFrame
print("\nThere are %d partitions in the voter_df DataFrame.\n" % voter_df.rdd.getNumPartitions())
print("\nThere are %d partitions in the voter_df_single DataFrame.\n" % voter_df_single.rdd.getNumPartitions())

# Add a ROW_ID field to each DataFrame
voter_df = voter_df.withColumn('ROW_ID', F.monotonically_increasing_id())
voter_df_single = voter_df_single.withColumn('ROW_ID', F.monotonically_increasing_id())

# Show the top 10 IDs in each DataFrame 
voter_df.orderBy(voter_df.ROW_ID.desc()).show(10)
voter_df_single.orderBy(voter_df_single.ROW_ID.desc()).show(10)


There are 2 partitions in the voter_df DataFrame.


There are 1 partitions in the voter_df_single DataFrame.

+--------------------+----------+
|          VOTER NAME|    ROW_ID|
+--------------------+----------+
|        Omar Narvaez|8589934609|
| Carolyn King Arnold|8589934608|
|       Mark  Clayton|8589934607|
|    Monica R. Alonzo|8589934606|
|         Erik Wilson|8589934605|
|       Scott  Griggs|8589934604|
|   the   final  2...|8589934603|
|  Philip T. Kingston|8589934602|
| the final 2018 A...|8589934601|
|   Tiffinni A. Young|8589934600|
+--------------------+----------+
only showing top 10 rows

+--------------------+------+
|          VOTER NAME|ROW_ID|
+--------------------+------+
|                NULL|    35|
|        Lee Kleinman|    34|
|  the  final  201...|    33|
|         Erik Wilson|    32|
|  the  final   20...|    31|
| Carolyn King Arnold|    30|
| Rickey D.  Callahan|    29|
|   the   final  2...|    28|
|    Monica R. Alonzo|    27|
|     Lee M. Kleinman|    2

In [77]:
voter_df = spark.read.format("csv").options(header=True, inferSchema=True).load("DallasCouncilVotes.csv.gz")

voter_df_march = (
    voter_df
        .filter(F.col("DATE").startswith("03/"))
        .select(F.col("VOTER NAME"))
        .withColumn("ROW_ID", F.monotonically_increasing_id())
).repartition(1)
voter_df_april = voter_df.filter(F.col("DATE").startswith("04/")).select(F.col("VOTER NAME")).repartition(1)

In [ ]:
# Determine the highest `ROW_ID` in `voter_df_march` and save it in the variable `previous_max_ID`.
previous_max_ID = voter_df_march.select('ROW_ID').rdd.max()[0]

# Add a `ROW_ID` column to `voter_df_april` starting at the value of `previous_max_ID` + 1.
voter_df_april = voter_df_april.withColumn('ROW_ID', F.monotonically_increasing_id() + previous_max_ID + 1)

# Show the `ROW_ID`'s from both DataFrames and compare
voter_df_april.select('ROW_ID').show()
voter_df_march.select('ROW_ID').show()

+------+
|ROW_ID|
+------+
|  1110|
|  1111|
|  1112|
|  1113|
|  1114|
|  1115|
|  1116|
|  1117|
|  1118|
|  1119|
|  1120|
|  1121|
|  1122|
|  1123|
|  1124|
|  1125|
|  1126|
|  1127|
|  1128|
|  1129|
+------+
only showing top 20 rows

+------+
|ROW_ID|
+------+
|     0|
|     1|
|     2|
|     3|
|     4|
|     5|
|     6|
|     7|
|     8|
|     9|
|    10|
|    11|
|    12|
|    13|
|    14|
|    15|
|    16|
|    17|
|    18|
|    19|
+------+
only showing top 20 rows

